# Indexing

An **index in SQL** is a special data structure that helps the database **find rows much faster**, just like the **index at the back of a book** helps you quickly find a topic instead of reading every page.

####used to retrive data faster and improve the query performance

Without an index, the database scans every row (**Full Table Scan**). With an index, it jumps directly to the required rows.

---

# What is an Index?

Suppose you have an `Employees` table with 10 million records.

| EmpID | Name  | Department | Salary |
| ----- | ----- | ---------- | ------ |
| 1     | Rahul | HR         | 40000  |
| 2     | Priya | IT         | 70000  |
| 3     | Arjun | Sales      | 50000  |
| ...   | ...   | ...        | ...    |

If you run:

```sql
SELECT * FROM Employees
WHERE EmpID = 100000;
```

### Without Index

The database checks:

```
1
2
3
4
...
99999
100000
```

This is called a **Full Table Scan**.

Time Complexity:

```
O(n)
```

---

### With Index

The database already has a sorted structure.

```
1
50
120
500
800
1000
...
100000
```

It directly reaches `100000`.

Time Complexity (approximately):

```
O(log n)
```

This is much faster.

---

# How Index Works

Most databases (SQL Server, MySQL InnoDB, PostgreSQL, Oracle) use a **B-Tree (Balanced Tree)** index.

Imagine this tree:

```
                500
             /       \
          250         750
        /    \       /    \
      100   400    600    900
```

If searching for **600**:

```
Start at 500
↓

600 > 500

↓

Go Right

↓

750

↓

600 < 750

↓

Go Left

↓

600 Found
```

Instead of checking every row, the database traverses the tree.

---

# Types of Indexes

## 1. Clustered Index

The table data itself is stored in the order of the index.

Example:

```
1
2
3
4
5
6
```

SQL Server allows only **one clustered index** per table because the data can only be physically ordered one way.

Example:

```sql
CREATE CLUSTERED INDEX IX_EmpID
ON Employees(EmpID);
```

### Best for

* Primary Key
* Range searches
* Sorting

Example:

```sql
SELECT *
FROM Employees
WHERE EmpID BETWEEN 100 AND 200;
```

Very fast.

---

## 2. Non-Clustered Index

The data stays where it is.

The index stores:

```
Name       Pointer

Amit  -----> Row 80
John  -----> Row 25
Rahul -----> Row 60
```

When searching:

```
Rahul

↓

Pointer

↓

Actual Row
```

Example:

```sql
CREATE INDEX IX_Name
ON Employees(Name);
```

---

## 3. Unique Index

Ensures duplicate values cannot exist.

Example:

```sql
CREATE UNIQUE INDEX IX_Email
ON Employees(Email);
```

Now:

```
abc@gmail.com
abc@gmail.com
```

Not allowed.

---

## 4. Composite Index (Multi-column Index)

Index created on multiple columns.

```sql
CREATE INDEX IX_DeptSalary
ON Employees(Department, Salary);
```

Works well for:

```sql
SELECT *
FROM Employees
WHERE Department='IT'
AND Salary > 60000;
```

### Important Rule

Order matters.

Index:

```
(Department, Salary)
```

Works for:

```sql
WHERE Department='IT'
```

and

```sql
WHERE Department='IT'
AND Salary>60000
```

But **not efficiently** for:

```sql
WHERE Salary>60000
```

because `Salary` is the second column.

---

## 5. Full-Text Index

Used for searching words inside text.

Example:

```sql
SELECT *
FROM Articles
WHERE CONTAINS(Content,'SQL');
```

Useful for:

* Google-like search
* Blogs
* Documents

---

## 6. Bitmap Index (Oracle)

Best for columns with very few distinct values.

Example:

```
Gender

Male
Female
```

or

```
Status

Active
Inactive
```

---

## 7. Hash Index

Mostly used in memory-based databases.

Excellent for:

```sql
WHERE EmpID = 500
```

Not suitable for:

```sql
BETWEEN
<
>
ORDER BY
```

---

# How to Create an Index

```sql
CREATE INDEX IX_Name
ON Employees(Name);
```

Multiple columns:

```sql
CREATE INDEX IX_DeptSalary
ON Employees(Department, Salary);
```

---

# How to Drop an Index

```sql
DROP INDEX IX_Name ON Employees;
```

---

# Real-Time Uses of Indexing

## Banking

Millions of customer accounts.

```sql
SELECT *
FROM Accounts
WHERE AccountNumber='123456789';
```

Without an index, every account would be scanned.

---

## E-commerce

Searching products:

```sql
SELECT *
FROM Products
WHERE ProductName='iPhone';
```

An index on `ProductName` speeds up searches.

---

## Social Media

Finding a user:

```sql
SELECT *
FROM Users
WHERE Username='john123';
```

Indexes allow quick lookup among millions of users.

---

## Hospital Systems

```sql
SELECT *
FROM Patients
WHERE PatientID=20001;
```

Index on `PatientID` makes retrieval fast.

---

## HR Systems

```sql
SELECT *
FROM Employees
WHERE Email='abc@gmail.com';
```

An index on `Email` avoids scanning the entire employee table.

---

## Online Shopping

```sql
SELECT *
FROM Orders
WHERE CustomerID=10;
```

An index on `CustomerID` speeds up fetching a customer's orders.

---

# Benefits of Indexing

| Benefit                  | Explanation                              |
| ------------------------ | ---------------------------------------- |
| Faster searches          | Quickly locates matching rows            |
| Faster `WHERE` queries   | Avoids full table scans                  |
| Faster `JOIN` operations | Joins on indexed columns are much faster |
| Faster `ORDER BY`        | Data may already be sorted               |
| Faster `GROUP BY`        | Reduces sorting effort in many cases     |
| Better performance       | Lower query execution time               |
| Improved scalability     | Handles large tables more efficiently    |

---

# Disadvantages of Indexing

Indexes are not always beneficial.

### 1. More Storage

Indexes require additional disk space.

---

### 2. Slower INSERT

When inserting a row:

```
Insert row
↓

Update every relevant index
```

More indexes mean more work.

---

### 3. Slower UPDATE

If an indexed column changes:

```sql
UPDATE Employees
SET Salary=80000
WHERE EmpID=10;
```

The corresponding index also needs updating.

---

### 4. Slower DELETE

Deleting rows requires removing entries from each related index.

---

# When Should You Create an Index?

Create indexes on columns that are:

* Frequently used in `WHERE` clauses
* Used in `JOIN` conditions
* Used in `ORDER BY`
* Used in `GROUP BY`
* Used for frequent searches
* Frequently referenced as foreign keys

Avoid indexing columns that:

* Have very few distinct values (unless using specialized index types such as bitmap indexes)
* Change very frequently
* Are in very small tables, where a full scan is often faster

---

# Interview Example

Suppose you have a table with **10 million employees**.

```sql
SELECT *
FROM Employees
WHERE EmpID = 5000000;
```

* **Without an index:** The database may scan all 10 million rows (full table scan).
* **With an index on `EmpID`:** It navigates the index structure to locate the matching row quickly.

---

## Summary

| Index Type    | Best Use                                | Example                |
| ------------- | --------------------------------------- | ---------------------- |
| Clustered     | Primary key, range queries              | `EmpID`                |
| Non-Clustered | Frequently searched columns             | `Name`, `Email`        |
| Unique        | Enforce uniqueness                      | `Email`, `Username`    |
| Composite     | Multi-column filtering                  | `(Department, Salary)` |
| Full-Text     | Search within large text                | Articles, documents    |
| Bitmap        | Low-cardinality columns (mainly Oracle) | `Gender`, `Status`     |
| Hash          | Fast equality lookups                   | `EmpID = 100`          |

**In real-world applications**, well-designed indexes can reduce query times from seconds to milliseconds, especially on tables with millions of rows. However, every additional index also increases the cost of `INSERT`, `UPDATE`, and `DELETE` operations, so indexes should be created based on actual query patterns rather than added to every column.



Absolutely. The best way to understand indexes is to use **one sample table** and create different indexes on it.

Let's use an **Employees** table.

## Sample Employees Table

| EmpID | Name   | Email                                       | Department | Salary | Gender | Status   |
| ----: | ------ | ------------------------------------------- | ---------- | -----: | ------ | -------- |
|   101 | Rahul  | [rahul@gmail.com](mailto:rahul@gmail.com)   | IT         |  70000 | M      | Active   |
|   102 | Priya  | [priya@gmail.com](mailto:priya@gmail.com)   | HR         |  45000 | F      | Active   |
|   103 | Amit   | [amit@gmail.com](mailto:amit@gmail.com)     | IT         |  80000 | M      | Inactive |
|   104 | Sneha  | [sneha@gmail.com](mailto:sneha@gmail.com)   | Sales      |  50000 | F      | Active   |
|   105 | John   | [john@gmail.com](mailto:john@gmail.com)     | IT         |  65000 | M      | Active   |
|   106 | David  | [david@gmail.com](mailto:david@gmail.com)   | Finance    |  90000 | M      | Active   |
|   107 | Anjali | [anjali@gmail.com](mailto:anjali@gmail.com) | HR         |  55000 | F      | Inactive |
|   108 | Kiran  | [kiran@gmail.com](mailto:kiran@gmail.com)   | IT         |  72000 | M      | Active   |

---

# 1. Clustered Index

Suppose `EmpID` is the clustered index.

```sql
CREATE CLUSTERED INDEX IX_EmpID
ON Employees(EmpID);
```

The database stores rows physically in `EmpID` order.

```
Disk Storage

101
102
103
104
105
106
107
108
```

Query

```sql
SELECT *
FROM Employees
WHERE EmpID = 105;
```

The database directly navigates to row **105**.

---

## Range Search

```sql
SELECT *
FROM Employees
WHERE EmpID BETWEEN 103 AND 106;
```

Database reads

```
103
104
105
106
```

No need to scan the whole table.

---

# 2. Non-Clustered Index

Create an index on `Name`.

```sql
CREATE INDEX IX_Name
ON Employees(Name);
```

The table stays the same, but a separate structure is created.

```
Index

Amit  ------> Row 103
Anjali ----> Row 107
David -----> Row 106
John ------> Row 105
Kiran -----> Row 108
Priya -----> Row 102
Rahul -----> Row 101
Sneha -----> Row 104
```

Query

```sql
SELECT *
FROM Employees
WHERE Name='John';
```

Database steps:

```
Search Index

↓

John

↓

Pointer

↓

Employee Row 105
```

---

# 3. Unique Index

Suppose Email must be unique.

```sql
CREATE UNIQUE INDEX IX_Email
ON Employees(Email);
```

Current data

| Email                                     |
| ----------------------------------------- |
| [rahul@gmail.com](mailto:rahul@gmail.com) |
| [priya@gmail.com](mailto:priya@gmail.com) |
| [amit@gmail.com](mailto:amit@gmail.com)   |

Trying to insert

```sql
INSERT INTO Employees
VALUES
(109,'Rahul','rahul@gmail.com','IT',75000,'M','Active');
```

Result

```
Error

Duplicate Email Not Allowed
```

---

# 4. Composite Index

Create

```sql
CREATE INDEX IX_DeptSalary
ON Employees(Department, Salary);
```

Index looks like

```
Department      Salary

Finance         90000
HR              45000
HR              55000
IT              65000
IT              70000
IT              72000
IT              80000
Sales           50000
```

Query

```sql
SELECT *
FROM Employees
WHERE Department='IT'
AND Salary >70000;
```

Database immediately jumps to

```
IT

↓

70000

↓

72000

80000
```

Only those rows are read.

---

## Bad Example

```sql
SELECT *
FROM Employees
WHERE Salary >70000;
```

Because the index starts with `Department`, the database cannot efficiently use the index for `Salary` alone.

---

# 5. Full-Text Index

Suppose we have an `Articles` table.

| Id | Title      | Content                        |
| -- | ---------- | ------------------------------ |
| 1  | SQL Basics | SQL Index improves performance |
| 2  | Oracle     | Oracle supports Bitmap Index   |
| 3  | MySQL      | B Tree Index is default        |

Create

```sql
CREATE FULLTEXT INDEX
ON Articles(Content);
```

Search

```sql
SELECT *
FROM Articles
WHERE CONTAINS(Content,'performance');
```

Result

| Id | Title      |
| -- | ---------- |
| 1  | SQL Basics |

Instead of scanning every article, the database searches the full-text index.

---

# 6. Bitmap Index (Oracle)

Suppose

| EmpID | Gender |
| ----- | ------ |
| 101   | M      |
| 102   | F      |
| 103   | M      |
| 104   | F      |
| 105   | M      |
| 106   | M      |
| 107   | F      |
| 108   | M      |

Bitmap representation

```
Gender=M

101 102 103 104 105 106 107 108

1   0   1   0   1   1   0   1
```

```
Gender=F

101 102 103 104 105 106 107 108

0   1   0   1   0   0   1   0
```

Query

```sql
SELECT *
FROM Employees
WHERE Gender='F';
```

Database uses the bitmap to quickly identify matching rows.

Best for columns with very few distinct values, such as:

* Gender
* Status
* Yes/No flags

---

# 7. Hash Index

Hash function example

```
EmpID

101

↓

Hash Function

↓

Bucket 1
```

```
102

↓

Bucket 5
```

```
103

↓

Bucket 3
```

Table

```
Bucket

1 → EmpID 101

2 → Empty

3 → EmpID 103

4 → EmpID 108

5 → EmpID 102
```

Query

```sql
SELECT *
FROM Employees
WHERE EmpID=103;
```

Database

```
Hash(103)

↓

Bucket 3

↓

Found
```

Very fast for equality searches.

Not suitable for

```sql
SELECT *
FROM Employees
WHERE EmpID BETWEEN 100 AND 105;
```

because hash indexes do not preserve order.

---

# Summary Table

| Index Type          | Sample Column          | Best Query                                     |
| ------------------- | ---------------------- | ---------------------------------------------- |
| **Clustered**       | `EmpID`                | `WHERE EmpID=101`, `BETWEEN`, `ORDER BY EmpID` |
| **Non-Clustered**   | `Name`                 | `WHERE Name='John'`                            |
| **Unique**          | `Email`                | Prevent duplicate emails                       |
| **Composite**       | `(Department, Salary)` | `WHERE Department='IT' AND Salary>70000`       |
| **Full-Text**       | `Content`              | Search words in text using `CONTAINS`          |
| **Bitmap** (Oracle) | `Gender`, `Status`     | Low-cardinality filters                        |
| **Hash**            | `EmpID`                | Fast equality lookups (`WHERE EmpID=103`)      |

### Which indexes are most common in real projects?

In day-to-day development with SQL Server, MySQL, or PostgreSQL, you'll most often use:

1. ✅ Clustered Index (usually on the primary key)
2. ✅ Non-Clustered Index
3. ✅ Composite Index
4. ✅ Unique Index

Full-text indexes are used when implementing search features, while bitmap and hash indexes are database-specific and used in more specialized scenarios.
